# NYU single-task ablations on the patch-mixing-off configuration (supplementary)

**Runs** (seed 42):
- Each run is the patch-mixing-off configuration of notebook 12 (the NYU final model, i.e. notebook 09 configuration + `frame_pe=False, photo_rng_fix=False`, with `use_patch_mix=False`) plus the ablated factor. The NYU final model itself keeps patch mixing on, so these runs are supplementary.
- **ST-D:** `w_seg=0`, depth only.
- **ST-S:** `w_depth=0`, segmentation only.
- **Baseline:** `results/final/nyu_base-nofpe-nopatchmix_s42`, seed 42 of notebook 12.


## Setup

***NOTE:*** The DINOv2 and DPT code is used straight from the local repositories `./dinov2` and `./DPT`.

In [1]:
import os, sys
if os.path.basename(os.getcwd()) == "experiments":
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

In [2]:
from mtl_final import *   # training code + decision rule, sigma, train_eval, load_eval (as in notebook 09)

configure(results_root=ROOT / "results")
CONFIG = load_json(FINAL_DIR / "config_final.json")
FINAL_TAG = "log" if CONFIG["depth_param"] == "log" else "base"
NYU_NOPM = dict(frame_pe=False, photo_rng_fix=False, use_patch_mix=False)   # PM-off configuration (adopted 2026-11-10, reverted 2026-11-11)
R_BASE = train_eval("nyu", 42, f"{FINAL_TAG}-nofpe-nopatchmix", CONFIG, **NYU_NOPM)   # cached (notebook 10 / 12): only loads
print(f"PM-off configuration, seed 42: RMSE {R_BASE['nyu_depth']['rmse']:.4f}, mIoU {R_BASE['nyu_seg']['mIoU']:.2f}")

sigma: {'rmse': 0.003, 'miou': 0.2, 'abs_rel_cv': 0.01, 'source': 'notebook 06 (raw: rmse=0.001669, miou=0.1389, abs_rel_cv=0.005997)'}
=== nyu base-nofpe-nopatchmix seed 42: /home/chrisjf/Documents/ml4cv/ckpt/final/nyu_base-nofpe-nopatchmix_s42
nyu_final: already trained
PM-off configuration, seed 42: RMSE 0.4021, mIoU 58.57


## Runs

In [3]:
RUNS = [("ST-D", "nopm-depthonly", dict(NYU_NOPM, w_seg=0.0)), ("ST-S", "nopm-segonly", dict(NYU_NOPM, w_depth=0.0))]
R_ST = {abl: train_eval("nyu", 42, f"{FINAL_TAG}-{suffix}", CONFIG, **extra) for abl, suffix, extra in RUNS}

=== nyu base-nopm-depthonly seed 42: /home/chrisjf/Documents/ml4cv/ckpt/final/nyu_base-nopm-depthonly_s42
nyu_final: already trained
=== nyu base-nopm-segonly seed 42: /home/chrisjf/Documents/ml4cv/ckpt/final/nyu_base-nopm-segonly_s42
nyu_final: already trained


## Results vs the PM-off configuration (seed 42)
Only the trained task is compared.

In [4]:
rows = []
for abl, R in R_ST.items():
    j = judge(abl, "nyu", R_BASE, R)
    d, b = R["nyu_depth"], R_BASE["nyu_depth"]
    if abl == "ST-D":
        rows.append(dict(ablation=abl, base_RMSE=b["rmse"], new_RMSE=d["rmse"], z_depth=j["z_depth"], base_AbsRel=b["abs_rel"], new_AbsRel=d["abs_rel"],
                         base_d1=b["d1"], new_d1=d["d1"], new_RMSE_flip=R["nyu_depth_flip"]["rmse"], log_bias=R["log_bias"]))
    else:
        rows.append(dict(ablation=abl, base_mIoU=R_BASE["nyu_seg"]["mIoU"], new_mIoU=R["nyu_seg"]["mIoU"], z_seg=j["z_seg"],
                         base_mIoU_per_image=R_BASE["nyu_seg_swinmtl"]["mIoU"], new_mIoU_per_image=R["nyu_seg_swinmtl"]["mIoU"]))
st = pd.DataFrame(rows)
(ROOT / "results" / "final").mkdir(parents=True, exist_ok=True)
st.to_csv(ROOT / "results" / "final" / "st_nopm_nyu.csv", index=False)
display(st.style.format(precision=4, na_rep="—"))

,ablation,base_RMSE,new_RMSE,z_depth,base_AbsRel,new_AbsRel,base_d1,new_d1,new_RMSE_flip,log_bias,base_mIoU,new_mIoU,z_seg,base_mIoU_per_image,new_mIoU_per_image
0,ST-D,0.4021,0.3787,7.7986,0.1150,0.1064,0.8798,0.8953,0.3743,0.0174,—,—,—,—,—
1,ST-S,—,—,—,—,—,—,—,—,—,58.5685,58.4334,-0.6755,57.6325,58.1412
